# Low-Light Image Enhancement – BSc Project

**Computer Vision, Bu-Ali Sina University, Winter 2026**

This notebook implements the three phases:
1. **Phase 1:** Image quality analysis and classification (handcrafted features + LR/SVM).
2. **Phase 2:** Classical enhancement (≥3 methods) + lightweight autoencoder.
3. **Phase 3:** Quantitative (PSNR, SSIM) and qualitative evaluation; discussion.

## Step 1: Environment and data check

Proposal: use **80–120 paired images** from LOL-v2 Real. We verify pair count and that images load.

In [ ]:
# Ensure package is importable: run from final-project root after: pip install -e .
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "low").exists():
    PROJECT_ROOT = PROJECT_ROOT / "final-project"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lowlight.config import DATA_DIR, MAX_PAIRS, MIN_PAIRS
from lowlight.data import discover_pairs, load_image

pairs, low_only, normal_only = discover_pairs(str(DATA_DIR))
n_pairs = len(pairs)
print(f"Total pairs: {n_pairs} (required: {MIN_PAIRS}–{MAX_PAIRS})")
print(f"OK: {MIN_PAIRS <= n_pairs <= MAX_PAIRS}")

if pairs:
    path_low, path_normal = pairs[0]
    sh_low = load_image(path_low).shape
    sh_norm = load_image(path_normal).shape
    print(f"Sample low shape: {sh_low}, normal shape: {sh_norm}")

## Milestone 1: Preprocessing (Proposal §3)

- **Resize:** standardize to 256×256.
- **Normalize:** scale pixel values to [0, 1].
- **Optional augmentation:** horizontal flip or small rotation (we use False by default; enable for training if justified in report).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lowlight.config import DATA_DIR, IMAGE_SIZE
from lowlight.data import load_dataset
from lowlight.utils import plot_pair

low_imgs, normal_imgs, pair_paths = load_dataset(
    str(DATA_DIR),
    target_size=IMAGE_SIZE,
    normalize=True,
    augment=False,
    seed=42,
)

print(f"Low images:    {low_imgs.shape}   dtype={low_imgs.dtype}   range=[{low_imgs.min():.2f}, {low_imgs.max():.2f}]")
print(f"Normal images: {normal_imgs.shape}   dtype={normal_imgs.dtype}   range=[{normal_imgs.min():.2f}, {normal_imgs.max():.2f}]")

In [ ]:
# Visual check: one pair (low vs normal)
plot_pair(low_imgs[0], normal_imgs[0], title="Preprocessed pair (resized 256×256, normalized [0,1])")